# EEG Emotion Recognition - Notebook 04: Model B (BiLSTM Baseline)

## Model B Architecture: Bidirectional LSTM
- **Purpose:** Captures long-range temporal dependencies in forward and backward directions across the EEG time-series.
- **Input:** `(batch_size, 256 timesteps, 32 channels)`
- **Key Layers:**
  - `Bidirectional(LSTM(64, return_sequences=True, dropout=0.2))`
  - `Bidirectional(LSTM(64, return_sequences=False, dropout=0.2))`
  - `Dense(128)` + Dropout(0.3) -> `Dense(64)` + Dropout(0.2) -> `Dense(4, linear)` predicting **[Valence, Arousal, Dominance, Liking]**.
- **Loss Function:** Huber Loss with Adam optimizer.

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/DEAP/processed'
    CHECKPOINT_DIR = '/content/drive/MyDrive/DEAP/checkpoints/bilstm'
else:
    DATA_DIR = '../data/processed' if os.path.exists('../data/processed') else './data/processed'
    CHECKPOINT_DIR = '../models/checkpoints/bilstm' if os.path.exists('../models') else './models/checkpoints/bilstm'

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import CONFIG
print("Setup completed successfully.")

In [ ]:
# Load Preprocessed Data Splits
train_data = np.load(os.path.join(DATA_DIR, 'train_data.npz'))
val_data = np.load(os.path.join(DATA_DIR, 'val_data.npz'))
test_data = np.load(os.path.join(DATA_DIR, 'test_data.npz'))

X_train, y_train = train_data['X'], train_data['y']
X_val, y_val = val_data['X'], val_data['y']
X_test, y_test = test_data['X'], test_data['y']

print(f"Training set:   X={X_train.shape}, y={y_train.shape}")
print(f"Validation set: X={X_val.shape}, y={y_val.shape}")
print(f"Testing set:    X={X_test.shape}, y={y_test.shape}")

In [ ]:
# Build and Compile Keras BiLSTM Model
try:
    import tensorflow as tf
    from src.models.bilstm import build_keras_bilstm
    
    model_bilstm = build_keras_bilstm(input_shape=(256, 32), num_outputs=4, lstm_units=64, dropout_rate=0.3)
    model_bilstm.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss=tf.keras.losses.Huber(delta=1.0),
        metrics=['mae', tf.keras.metrics.RootMeanSquaredError(name='rmse')]
    )
    model_bilstm.summary()
except ImportError:
    print("TensorFlow not installed in current environment. Ready for Google Colab.")

In [ ]:
# Train Model B (BiLSTM)
if 'model_bilstm' in locals():
    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6),
        tf.keras.callbacks.ModelCheckpoint(os.path.join(CHECKPOINT_DIR, 'best_bilstm_model.keras'), save_best_only=True)
    ]
    
    history = model_bilstm.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        batch_size=CONFIG.training.batch_size,
        epochs=CONFIG.training.epochs,
        callbacks=callbacks,
        verbose=1
    )